# LeetCode 912 — Sort an Array bằng Python

Link đề: https://leetcode.com/problems/sort-an-array/ · Độ khó: **Medium** · Chủ đề: Array (mảng), Divide and Conquer (chia để trị), Sorting (sắp xếp), Heap (Priority Queue), Merge Sort, Bucket Sort, Radix Sort, Counting Sort

Skeleton LeetCode:

```python
class Solution:
    def sortArray(self, nums: List[int]) -> List[int]:
        ...
```

Ghi chú: đề yêu cầu sắp xếp mảng tăng dần **không dùng hàm có sẵn**, trong thời gian `O(n log n)` và **bộ nhớ phụ nhỏ nhất có thể**. Vì vậy ở **mọi cách nộp** trong notebook này ta **không dùng** `sorted`, `list.sort`, `heapq`, `bisect`... Hàm `sorted` chỉ xuất hiện trong **phần kiểm thử** (làm **oracle**) và trong các bài tập đối chiếu. Đây là cơ hội học bốn thuật toán sắp xếp kinh điển: **sắp xếp chèn** (`O(n^2)`), **trộn (merge sort)**, **nhanh (quick sort)**, **vun đống (heap sort)** và một thuật toán không so sánh là **đếm (counting sort)**.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: sắp xếp mảng số nguyên (có thể trùng, có thể âm) tăng dần mà không dùng hàm sắp xếp có sẵn.
2. Nắm các công cụ Python cần thiết: hoán đổi `a[i], a[j] = a[j], a[i]`, gán lát cắt `a[lo:hi] = tmp[lo:hi]`, đệ quy có giới hạn độ sâu, vòng `while` với nhiều con trỏ, và chỉ số cây đống (`2i + 1`, `2i + 2`, `(i - 1) // 2`).
3. Hiểu nguyên lý của từng thuật toán: **chèn** (mở rộng đoạn đã sắp xếp), **trộn** (trộn hai nửa đã sắp xếp), **nhanh** (chia theo chốt, chia ba phần để xử lý số trùng), **vun đống** (đống cực đại và lấy phần tử lớn nhất ra cuối), **đếm** (đếm số lần xuất hiện của mỗi giá trị trong miền nhỏ).
4. Đi qua 5 cách và so sánh: độ phức tạp thời gian trường hợp xấu nhất, bộ nhớ phụ, tính ổn định (stable) và hằng số thực tế.
5. Tránh các lỗi kinh điển: quick sort chọn chốt cố định trên dữ liệu đã sắp xếp (`O(n^2)` và tràn đệ quy), thiếu phần dư khi trộn, sai biên đống khi vun đống, quên dịch chỉ số âm khi đếm, và dùng thuật toán `O(n^2)` với `n = 5 * 10^4`.

## 2. Đề bài & ví dụ

Cho mảng số nguyên `nums`, hãy sắp xếp mảng theo thứ tự **tăng dần** rồi trả về nó.

Bạn phải giải bài toán **không dùng bất kỳ hàm có sẵn nào** (hàm sắp xếp), trong thời gian `O(n log n)` và với **bộ nhớ phụ nhỏ nhất có thể**.

**Ví dụ 1**

- Input: `nums = [5,2,3,1]`
- Output: `[1,2,3,5]`

**Ví dụ 2**

- Input: `nums = [5,1,1,2,0,0]`
- Output: `[0,0,1,1,2,5]` (giá trị không nhất thiết khác nhau)

**Ràng buộc:**

- `1 <= nums.length <= 5 * 10^4`
- `-5 * 10^4 <= nums[i] <= 5 * 10^4`

In [1]:
from typing import List
from collections import Counter
import random, sys, time

examples = [
    ([5, 2, 3, 1], [1, 2, 3, 5]),
    ([5, 1, 1, 2, 0, 0], [0, 0, 1, 1, 2, 5]),
]
for nums, expected in examples:
    print(f"nums={nums} kỳ vọng={expected}")

nums=[5, 2, 3, 1] kỳ vọng=[1, 2, 3, 5]
nums=[5, 1, 1, 2, 0, 0] kỳ vọng=[0, 0, 1, 1, 2, 5]


## 3. Kiến thức nền

### 3.1. Kiểm tra "đã sắp xếp" và "là hoán vị" mà không dùng `sorted`

Một mảng `b` là kết quả sắp xếp đúng của `a` khi và chỉ khi (1) `b` **không giảm**: `b[i] <= b[i + 1]` với mọi `i`, và (2) `b` là **hoán vị** của `a`: `Counter(b) == Counter(a)`. Hai điều kiện này **độc lập** với mọi thuật toán sắp xếp nên dùng làm bộ kiểm tra thứ hai bên cạnh oracle `sorted`.

In [2]:
def is_nondecreasing(b: List[int]) -> bool:
    return all(b[i] <= b[i + 1] for i in range(len(b) - 1))

def is_valid_sort(a: List[int], b: List[int]) -> bool:
    return len(a) == len(b) and is_nondecreasing(b) and Counter(a) == Counter(b)

print(is_valid_sort([5, 2, 3, 1], [1, 2, 3, 5]), is_valid_sort([5, 2, 3, 1], [1, 2, 3, 3]), is_valid_sort([5, 2, 3, 1], [1, 3, 2, 5]))

True False False


### 3.2. Trộn hai đoạn đã sắp xếp

Nếu `L` và `R` đã sắp xếp, ta trộn bằng hai con trỏ `i`, `j`: so sánh `L[i]` với `R[j]`, lấy cái nhỏ hơn (khi bằng nhau lấy bên **trái** để giữ tính **ổn định**), tăng con trỏ tương ứng; khi một bên hết thì **chép phần dư của bên kia**.

In [3]:
def merge(L: List[int], R: List[int]) -> List[int]:
    out, i, j = [], 0, 0
    while i < len(L) and j < len(R):
        if L[i] <= R[j]:
            out.append(L[i]); i += 1
        else:
            out.append(R[j]); j += 1
    out.extend(L[i:])
    out.extend(R[j:])
    return out

print(merge([1, 4], [2, 3]), merge([2, 2], [1, 2, 3]), merge([], [1]))

[1, 2, 3, 4] [1, 2, 2, 2, 3] [1]


### 3.3. Đống nhị phân lưu trong mảng

Với chỉ số bắt đầu từ `0`: nút `i` có **con trái** `2i + 1`, **con phải** `2i + 2`, **cha** `(i - 1) // 2`. **Đống cực đại**: mỗi nút `>=` hai con. `sift_down(i, n)` đẩy `a[i]` xuống cho tới khi thỏa tính chất đống trong `a[0..n-1]`.

In [4]:
i = 3
print("cha của", i, "là", (i - 1) // 2, "| con của", i, "là", 2 * i + 1, 2 * i + 2)
for i in range(7):
    print(f"nút {i}: cha {(i - 1) // 2 if i else '-'}, con {2 * i + 1}, {2 * i + 2}")

cha của 3 là 1 | con của 3 là 7 8
nút 0: cha -, con 1, 2
nút 1: cha 0, con 3, 4
nút 2: cha 0, con 5, 6
nút 3: cha 1, con 7, 8
nút 4: cha 1, con 9, 10
nút 5: cha 2, con 11, 12
nút 6: cha 2, con 13, 14


### 3.4. Miền giá trị nhỏ cho phép đếm

Vì `-5 * 10^4 <= nums[i] <= 5 * 10^4` chỉ có `100001` giá trị, ta có thể **đếm** số lần xuất hiện của mỗi giá trị trong mảng `cnt` có `100001` phần tử (chỉ số `x + 50000`) rồi in ra theo thứ tự giá trị tăng dần: thời gian `O(n + R)` với `R = 100001`.

In [5]:
OFFSET, R = 50000, 100001
cnt = [0] * R
for x in [5, 1, 1, 2, 0, 0, -3]:
    cnt[x + OFFSET] += 1
print({v - OFFSET: c for v, c in enumerate(cnt) if c})

{-3: 1, 0: 2, 1: 2, 2: 1, 5: 1}


## 4. Ý tưởng nền tảng / trực giác (tính tay)

Sắp xếp `[5, 2, 3, 1]`:

- **Chèn:** `[5] | 2 -> [2,5] | 3 -> [2,3,5] | 1 -> [1,2,3,5]`.
- **Trộn:** chia `[5,2] [3,1]`; sắp xếp từng nửa `[2,5] [1,3]`; trộn `[1,2,3,5]`.
- **Nhanh:** chọn chốt `3`: nhỏ hơn `[2,1]`, bằng `[3]`, lớn hơn `[5]`; sắp xếp nhỏ `[1,2]`; ghép `[1,2,3,5]`.
- **Vun đống:** dựng đống cực đại `[5,2,3,1]` (đã là đống), đưa `5` xuống cuối, vun lại, đưa `3` xuống, ... ra `[1,2,3,5]`.
- **Đếm:** `cnt[1]=cnt[2]=cnt[3]=cnt[5]=1`, in theo thứ tự `1, 2, 3, 5`.

## 5. Cách 1 — Brute Force: sắp xếp chèn (`O(n^2)`)

**Ý tưởng.** Duy trì đoạn đầu `a[0..i-1]` đã sắp xếp; lấy `key = a[i]` và **dịch phải** các phần tử lớn hơn `key` trong đoạn đó, rồi đặt `key` vào chỗ trống. Đơn giản, ổn định, bộ nhớ phụ `O(1)`, nhưng `O(n^2)` trường hợp xấu nhất.

In [6]:
def insertion_sort(nums: List[int]) -> List[int]:
    a = nums
    for i in range(1, len(a)):
        key = a[i]
        j = i - 1
        while j >= 0 and a[j] > key:
            a[j + 1] = a[j]
            j -= 1
        a[j + 1] = key
    return a

for nums, expected in examples:
    got = insertion_sort(list(nums))
    print(nums, "->", got, "OK" if got == expected else "SAI")
    assert got == expected

[5, 2, 3, 1] -> [1, 2, 3, 5] OK
[5, 1, 1, 2, 0, 0] -> [0, 0, 1, 1, 2, 5] OK


**Độ phức tạp và nút thắt.** Trung bình và xấu nhất `O(n^2)` (mỗi phần tử có thể dịch tới `i` bước); với `n = 5 * 10^4` là `~ 10^9` thao tác: **TLE**. Xem thời gian tăng gấp bốn lần khi `n` tăng gấp đôi:

In [7]:
rng = random.Random(912)
for n in (500, 1000, 2000, 4000):
    nums = [rng.randint(-50000, 50000) for _ in range(n)]
    t0 = time.perf_counter()
    insertion_sort(nums)
    print(f"n={n:5}: {1000 * (time.perf_counter() - t0):9.1f} ms")

n=  500:       3.8 ms
n= 1000:      17.5 ms
n= 2000:      75.1 ms


n= 4000:     293.0 ms


Nút thắt: sắp xếp chèn **dịch từng bước một**; thuật toán `O(n log n)` chia nhỏ bài toán hoặc dùng cấu trúc đống để tránh phép dịch dài.

## 6. Ý tưởng thuật toán / bước đột phá

Có nhiều cách đạt `O(n log n)`:

- **Chia để trị:** sắp xếp từng nửa rồi **trộn** (merge sort), hoặc **chia theo chốt** rồi sắp xếp hai phần (quick sort).
- **Cấu trúc đống:** dựng đống rồi lần lượt lấy phần tử lớn nhất đặt vào cuối (heap sort): bộ nhớ phụ `O(1)`, trường hợp xấu nhất vẫn `O(n log n)`.
- **Tận dụng miền giá trị nhỏ:** đếm số lần xuất hiện (counting sort) `O(n + R)`.

| Cách | Thời gian xấu nhất | Bộ nhớ phụ | Ổn định |
|---|---|---|---|
| 1. Chèn | `O(n^2)` | `O(1)` | có |
| 2. Trộn | `O(n log n)` | `O(n)` | có |
| 3. Nhanh (3 phần, chốt ngẫu nhiên) | `O(n log n)` kỳ vọng | `O(log n)` | không |
| 4. Vun đống | `O(n log n)` | `O(1)` | không |
| 5. Đếm | `O(n + R)` | `O(R)` | có (khi viết thêm) |

## 7. Các cách tiếp theo

### 7.1. Cách 2 — Sắp xếp trộn (merge sort)

**Ý tưởng.** `sort(lo, hi)` sắp xếp đoạn `[lo, hi)`: nếu có `<= 1` phần tử thì xong; ngược lại chia đôi tại `mid`, sắp xếp từng nửa, rồi **trộn** hai nửa vào mảng phụ `tmp` và chép lại.

**Giải thích từng đoạn.**

- Dùng **một** mảng phụ `tmp` cho cả quá trình (`O(n)` bộ nhớ) thay vì cấp phát mới ở mỗi lần trộn.
- Trộn: so sánh `nums[i] <= nums[j]` (dấu `<=` giữ ổn định), sau đó chép phần dư của nửa trái và nửa phải.
- Chiều sâu đệ quy `O(log n)`.

In [8]:
def merge_sort(nums: List[int]) -> List[int]:
    n = len(nums)
    tmp = [0] * n

    def sort(lo: int, hi: int) -> None:
        if hi - lo <= 1:
            return
        mid = (lo + hi) // 2
        sort(lo, mid)
        sort(mid, hi)
        i, j, k = lo, mid, lo
        while i < mid and j < hi:
            if nums[i] <= nums[j]:
                tmp[k] = nums[i]
                i += 1
            else:
                tmp[k] = nums[j]
                j += 1
            k += 1
        while i < mid:
            tmp[k] = nums[i]
            i += 1
            k += 1
        while j < hi:
            tmp[k] = nums[j]
            j += 1
            k += 1
        nums[lo:hi] = tmp[lo:hi]

    sort(0, n)
    return nums

for nums, expected in examples:
    got = merge_sort(list(nums))
    print(nums, "->", got, "OK" if got == expected else "SAI")
    assert got == expected

[5, 2, 3, 1] -> [1, 2, 3, 5] OK
[5, 1, 1, 2, 0, 0] -> [0, 0, 1, 1, 2, 5] OK


**Code nộp LeetCode (Cách 2):**

In [9]:
class Solution:
    def sortArray(self, nums: List[int]) -> List[int]:
        n = len(nums)
        tmp = [0] * n

        def sort(lo: int, hi: int) -> None:
            if hi - lo <= 1:
                return
            mid = (lo + hi) // 2
            sort(lo, mid)
            sort(mid, hi)
            i, j, k = lo, mid, lo
            while i < mid and j < hi:
                if nums[i] <= nums[j]:
                    tmp[k] = nums[i]
                    i += 1
                else:
                    tmp[k] = nums[j]
                    j += 1
                k += 1
            while i < mid:
                tmp[k] = nums[i]
                i += 1
                k += 1
            while j < hi:
                tmp[k] = nums[j]
                j += 1
                k += 1
            nums[lo:hi] = tmp[lo:hi]

        sort(0, n)
        return nums

sol_merge = Solution().sortArray
assert [sol_merge(list(n)) for n, _ in examples] == [e for _, e in examples]
print("Cách 2 qua ví dụ")

Cách 2 qua ví dụ


**Độ phức tạp (Cách 2):** thời gian `O(n log n)` trong mọi trường hợp; bộ nhớ phụ `O(n)` (mảng `tmp`) cộng `O(log n)` ngăn xếp đệ quy; ổn định.

---

### 7.2. Cách 3 — Sắp xếp nhanh (quick sort) chia ba phần với chốt ngẫu nhiên

**Ý tưởng.** Chọn một **chốt** `p`, chia đoạn thành ba phần: `< p`, `== p`, `> p` (thuật toán **cờ Hà Lan**, Dutch national flag), rồi sắp xếp đệ quy hai phần `<` và `>`; phần `== p` đã đúng chỗ. Chia ba phần giúp xử lý **nhiều số trùng** (không rơi vào `O(n^2)`); chốt **ngẫu nhiên** tránh dữ liệu xấu cố định.

**Giải thích từng đoạn.**

- `lt, i, gt = lo, lo, hi`: `a[lo..lt-1] < p`, `a[lt..i-1] == p`, `a[gt+1..hi] > p`, `a[i..gt]` chưa phân loại.
- Nếu `a[i] < p`: đổi chỗ với `a[lt]`, `lt += 1`, `i += 1`; nếu `a[i] > p`: đổi với `a[gt]`, `gt -= 1` (không tăng `i`); ngược lại `i += 1`.
- Đệ quy vào phần **nhỏ hơn**, lặp (vòng `while`) cho phần lớn hơn để chiều sâu đệ quy chỉ `O(log n)`.

In [10]:
def quick_sort(nums: List[int]) -> List[int]:
    a = nums

    def qs(lo: int, hi: int) -> None:
        while lo < hi:
            p = a[random.randint(lo, hi)]
            lt, i, gt = lo, lo, hi
            while i <= gt:
                if a[i] < p:
                    a[lt], a[i] = a[i], a[lt]
                    lt += 1
                    i += 1
                elif a[i] > p:
                    a[i], a[gt] = a[gt], a[i]
                    gt -= 1
                else:
                    i += 1
            if lt - lo < hi - gt:
                qs(lo, lt - 1)
                lo = gt + 1
            else:
                qs(gt + 1, hi)
                hi = lt - 1

    qs(0, len(a) - 1)
    return a

for nums, expected in examples:
    got = quick_sort(list(nums))
    print(nums, "->", got, "OK" if got == expected else "SAI")
    assert got == expected

[5, 2, 3, 1] -> [1, 2, 3, 5] OK
[5, 1, 1, 2, 0, 0] -> [0, 0, 1, 1, 2, 5] OK


**Code nộp LeetCode (Cách 3):**

In [11]:
class Solution:
    def sortArray(self, nums: List[int]) -> List[int]:
        import random
        a = nums

        def qs(lo: int, hi: int) -> None:
            while lo < hi:
                p = a[random.randint(lo, hi)]
                lt, i, gt = lo, lo, hi
                while i <= gt:
                    if a[i] < p:
                        a[lt], a[i] = a[i], a[lt]
                        lt += 1
                        i += 1
                    elif a[i] > p:
                        a[i], a[gt] = a[gt], a[i]
                        gt -= 1
                    else:
                        i += 1
                if lt - lo < hi - gt:
                    qs(lo, lt - 1)
                    lo = gt + 1
                else:
                    qs(gt + 1, hi)
                    hi = lt - 1

        qs(0, len(a) - 1)
        return a

sol_quick = Solution().sortArray
assert [sol_quick(list(n)) for n, _ in examples] == [e for _, e in examples]
print("Cách 3 qua ví dụ")

Cách 3 qua ví dụ


**Độ phức tạp (Cách 3):** thời gian `O(n log n)` **kỳ vọng** (chốt ngẫu nhiên), `O(n^2)` trong trường hợp xấu cực kỳ hiếm; bộ nhớ phụ `O(log n)` (nhờ đệ quy vào phần nhỏ hơn); **không ổn định**.

---

### 7.3. Cách 4 — Sắp xếp vun đống (heap sort)

**Ý tưởng.** Xem mảng là đống nhị phân. **Dựng đống cực đại** (từ nút trong cuối cùng `n//2 - 1` ngược về `0`, gọi `sift_down`); sau đó lặp: đổi `a[0]` (lớn nhất) với `a[end]`, giảm kích thước đống `end`, rồi `sift_down(0)` để khôi phục. Mảng thu được tăng dần.

**Giải thích từng đoạn.**

- `sift_down(i, size)`: chọn con lớn hơn `c` (nếu có); nếu `a[c] > a[i]` thì đổi chỗ và tiếp tục xuống, ngược lại dừng.
- Biên của đống phải là `size` (không phải `len(a)`): phần đuôi từ `size` trở đi đã sắp xếp và không được đụng tới.
- Không dùng đệ quy, bộ nhớ phụ `O(1)`.

In [12]:
def heap_sort(nums: List[int]) -> List[int]:
    a = nums
    n = len(a)

    def sift_down(i: int, size: int) -> None:
        while True:
            left = 2 * i + 1
            if left >= size:
                return
            c = left
            if left + 1 < size and a[left + 1] > a[left]:
                c = left + 1
            if a[c] > a[i]:
                a[c], a[i] = a[i], a[c]
                i = c
            else:
                return

    for i in range(n // 2 - 1, -1, -1):
        sift_down(i, n)
    for end in range(n - 1, 0, -1):
        a[0], a[end] = a[end], a[0]
        sift_down(0, end)
    return a

for nums, expected in examples:
    got = heap_sort(list(nums))
    print(nums, "->", got, "OK" if got == expected else "SAI")
    assert got == expected

[5, 2, 3, 1] -> [1, 2, 3, 5] OK
[5, 1, 1, 2, 0, 0] -> [0, 0, 1, 1, 2, 5] OK


**Code nộp LeetCode (Cách 4):**

In [13]:
class Solution:
    def sortArray(self, nums: List[int]) -> List[int]:
        n = len(nums)

        def sift_down(i: int, size: int) -> None:
            while True:
                left = 2 * i + 1
                if left >= size:
                    return
                c = left
                if left + 1 < size and nums[left + 1] > nums[left]:
                    c = left + 1
                if nums[c] > nums[i]:
                    nums[c], nums[i] = nums[i], nums[c]
                    i = c
                else:
                    return

        for i in range(n // 2 - 1, -1, -1):
            sift_down(i, n)
        for end in range(n - 1, 0, -1):
            nums[0], nums[end] = nums[end], nums[0]
            sift_down(0, end)
        return nums

sol_heap = Solution().sortArray
assert [sol_heap(list(n)) for n, _ in examples] == [e for _, e in examples]
print("Cách 4 qua ví dụ")

Cách 4 qua ví dụ


**Độ phức tạp (Cách 4):** dựng đống `O(n)`; `n - 1` lần `sift_down` mỗi lần `O(log n)`: tổng `O(n log n)` trong **mọi** trường hợp; bộ nhớ phụ `O(1)`; **không ổn định**.

---

### 7.4. Cách 5 — Sắp xếp đếm (counting sort)

**Ý tưởng.** Miền giá trị chỉ `100001` giá trị nên **đếm** số lần xuất hiện: `cnt[x + 50000] += 1`; rồi ghi lại mảng theo giá trị tăng dần. Không so sánh phần tử với nhau.

**Giải thích từng đoạn.**

- `OFFSET = 50000` đưa giá trị âm về chỉ số không âm.
- Ghi lại: với mỗi `v` từ `0` đến `100000`, ghi `cnt[v]` lần giá trị `v - 50000` vào `nums` theo thứ tự.

In [14]:
def counting_sort(nums: List[int]) -> List[int]:
    OFFSET, R = 50000, 100001
    cnt = [0] * R
    for x in nums:
        cnt[x + OFFSET] += 1
    i = 0
    for v in range(R):
        c = cnt[v]
        while c:
            nums[i] = v - OFFSET
            i += 1
            c -= 1
    return nums

for nums, expected in examples:
    got = counting_sort(list(nums))
    print(nums, "->", got, "OK" if got == expected else "SAI")
    assert got == expected

[5, 2, 3, 1] -> [1, 2, 3, 5] OK
[5, 1, 1, 2, 0, 0] -> [0, 0, 1, 1, 2, 5] OK


**Code nộp LeetCode (Cách 5):**

In [15]:
class Solution:
    def sortArray(self, nums: List[int]) -> List[int]:
        OFFSET, R = 50000, 100001
        cnt = [0] * R
        for x in nums:
            cnt[x + OFFSET] += 1
        i = 0
        for v in range(R):
            c = cnt[v]
            while c:
                nums[i] = v - OFFSET
                i += 1
                c -= 1
        return nums

sol_count = Solution().sortArray
assert [sol_count(list(n)) for n, _ in examples] == [e for _, e in examples]
print("Cách 5 qua ví dụ")

Cách 5 qua ví dụ


**Độ phức tạp (Cách 5):** `O(n + R)` thời gian với `R = 100001`; bộ nhớ phụ `O(R)` (mảng `cnt`). Rất nhanh khi miền nhỏ, nhưng **không dùng được** nếu giá trị lớn tùy ý.

## 8. TRACE từng bước

### 8.1. Sắp xếp trộn cho `[5, 2, 3, 1]`

In [16]:
def trace_merge(nums):
    n = len(nums)
    tmp = [0] * n
    def sort(lo, hi, depth=0):
        pad = "  " * depth
        if hi - lo <= 1:
            return
        mid = (lo + hi) // 2
        print(f"{pad}chia [{lo},{hi}) = {nums[lo:hi]} thành {nums[lo:mid]} và {nums[mid:hi]}")
        sort(lo, mid, depth + 1)
        sort(mid, hi, depth + 1)
        i, j, k = lo, mid, lo
        while i < mid and j < hi:
            if nums[i] <= nums[j]:
                tmp[k] = nums[i]; i += 1
            else:
                tmp[k] = nums[j]; j += 1
            k += 1
        while i < mid:
            tmp[k] = nums[i]; i += 1; k += 1
        while j < hi:
            tmp[k] = nums[j]; j += 1; k += 1
        nums[lo:hi] = tmp[lo:hi]
        print(f"{pad}trộn -> {nums[lo:hi]}")
    sort(0, n)
    return nums

assert trace_merge([5, 2, 3, 1]) == [1, 2, 3, 5]

chia [0,4) = [5, 2, 3, 1] thành [5, 2] và [3, 1]
  chia [0,2) = [5, 2] thành [5] và [2]
  trộn -> [2, 5]
  chia [2,4) = [3, 1] thành [3] và [1]
  trộn -> [1, 3]
trộn -> [1, 2, 3, 5]


### 8.2. Chia ba phần (quick sort) cho `[5, 1, 1, 2, 0, 0]` với chốt `1`

In [17]:
def trace_partition(a, p):
    a = list(a)
    lt, i, gt = 0, 0, len(a) - 1
    print(f"chốt {p}: ban đầu {a}")
    while i <= gt:
        if a[i] < p:
            a[lt], a[i] = a[i], a[lt]; lt += 1; i += 1
            action = "nhỏ hơn: đổi với lt"
        elif a[i] > p:
            a[i], a[gt] = a[gt], a[i]; gt -= 1
            action = "lớn hơn: đổi với gt"
        else:
            i += 1
            action = "bằng chốt: bỏ qua"
        print(f"  {action:22} -> {a} (lt={lt}, i={i}, gt={gt})")
    print(f"kết quả: < chốt: {a[:lt]}, == chốt: {a[lt:gt + 1]}, > chốt: {a[gt + 1:]}")
    return a, lt, gt

a, lt, gt = trace_partition([5, 1, 1, 2, 0, 0], 1)
assert sorted(a[:lt]) == [0, 0] and a[lt:gt + 1] == [1, 1] and sorted(a[gt + 1:]) == [2, 5]

chốt 1: ban đầu [5, 1, 1, 2, 0, 0]
  lớn hơn: đổi với gt    -> [0, 1, 1, 2, 0, 5] (lt=0, i=0, gt=4)
  nhỏ hơn: đổi với lt    -> [0, 1, 1, 2, 0, 5] (lt=1, i=1, gt=4)
  bằng chốt: bỏ qua      -> [0, 1, 1, 2, 0, 5] (lt=1, i=2, gt=4)
  bằng chốt: bỏ qua      -> [0, 1, 1, 2, 0, 5] (lt=1, i=3, gt=4)
  lớn hơn: đổi với gt    -> [0, 1, 1, 0, 2, 5] (lt=1, i=3, gt=3)
  nhỏ hơn: đổi với lt    -> [0, 0, 1, 1, 2, 5] (lt=2, i=4, gt=3)
kết quả: < chốt: [0, 0], == chốt: [1, 1], > chốt: [2, 5]


### 8.3. Vun đống cho `[5, 2, 3, 1]`

Dựng đống: `n = 4`, nút trong cuối cùng là `4 // 2 - 1 = 1`; `sift_down(1)`: `a[1] = 2`, con `a[3] = 1` nhỏ hơn nên giữ; `sift_down(0)`: `5 >= 2, 3` giữ. Đống: `[5, 2, 3, 1]`. Sau đó: đổi `a[0]` với `a[3]` thành `[1, 2, 3, 5]`, vun lại phần `[0..2]` thành `[3, 2, 1, 5]`; đổi `a[0]` với `a[2]` thành `[1, 2, 3, 5]`, vun `[0..1]` thành `[2, 1, 3, 5]`; đổi `a[0]` với `a[1]` thành `[1, 2, 3, 5]`.

In [18]:
def trace_heap(a):
    a = list(a)
    n = len(a)
    def sift_down(i, size):
        while True:
            left = 2 * i + 1
            if left >= size:
                return
            c = left
            if left + 1 < size and a[left + 1] > a[left]:
                c = left + 1
            if a[c] > a[i]:
                a[c], a[i] = a[i], a[c]
                i = c
            else:
                return
    for i in range(n // 2 - 1, -1, -1):
        sift_down(i, n)
    print("sau khi dựng đống cực đại:", a)
    for end in range(n - 1, 0, -1):
        a[0], a[end] = a[end], a[0]
        sift_down(0, end)
        print(f"đưa lớn nhất ra a[{end}] -> {a} (đống còn {end} phần tử)")
    return a

assert trace_heap([5, 2, 3, 1]) == [1, 2, 3, 5]

sau khi dựng đống cực đại: [5, 2, 3, 1]
đưa lớn nhất ra a[3] -> [3, 2, 1, 5] (đống còn 3 phần tử)
đưa lớn nhất ra a[2] -> [2, 1, 3, 5] (đống còn 2 phần tử)
đưa lớn nhất ra a[1] -> [1, 2, 3, 5] (đống còn 1 phần tử)


## 9. Code nộp LeetCode

Đề yêu cầu `O(n log n)` **và bộ nhớ phụ nhỏ nhất có thể**. **Sắp xếp vun đống** đạt cả hai: `O(n log n)` trong mọi trường hợp và `O(1)` bộ nhớ phụ, không cần đệ quy; ta chọn **Cách 4**.

```python
class Solution:
    def sortArray(self, nums: List[int]) -> List[int]:
        n = len(nums)

        def sift_down(i: int, size: int) -> None:
            while True:
                left = 2 * i + 1
                if left >= size:
                    return
                c = left
                if left + 1 < size and nums[left + 1] > nums[left]:
                    c = left + 1
                if nums[c] > nums[i]:
                    nums[c], nums[i] = nums[i], nums[c]
                    i = c
                else:
                    return

        for i in range(n // 2 - 1, -1, -1):
            sift_down(i, n)
        for end in range(n - 1, 0, -1):
            nums[0], nums[end] = nums[end], nums[0]
            sift_down(0, end)
        return nums
```

In [19]:
class Solution:
    def sortArray(self, nums: List[int]) -> List[int]:
        n = len(nums)

        def sift_down(i: int, size: int) -> None:
            while True:
                left = 2 * i + 1
                if left >= size:
                    return
                c = left
                if left + 1 < size and nums[left + 1] > nums[left]:
                    c = left + 1
                if nums[c] > nums[i]:
                    nums[c], nums[i] = nums[i], nums[c]
                    i = c
                else:
                    return

        for i in range(n // 2 - 1, -1, -1):
            sift_down(i, n)
        for end in range(n - 1, 0, -1):
            nums[0], nums[end] = nums[end], nums[0]
            sift_down(0, end)
        return nums

sol = Solution()
for nums, expected in examples:
    got = sol.sortArray(list(nums))
    print(nums, "->", got, "OK" if got == expected else "SAI")
    assert got == expected

[5, 2, 3, 1] -> [1, 2, 3, 5] OK
[5, 1, 1, 2, 0, 0] -> [0, 0, 1, 1, 2, 5] OK


## 10. Vì sao đúng

**(a) Sắp xếp chèn.** Bất biến vòng lặp: trước vòng `i`, `a[0..i-1]` là các phần tử ban đầu của đoạn đó đã sắp xếp. Dịch các phần tử `> key` sang phải một ô rồi đặt `key` vào đúng chỗ giữ `a[0..i]` đã sắp xếp. Cuối cùng `i = n`.

**(b) Sắp xếp trộn.** Quy nạp theo độ dài: đoạn `<= 1` phần tử hiển nhiên đã sắp xếp; hai nửa sau khi sắp xếp đệ quy đã tăng dần; bước trộn hai con trỏ luôn lấy phần tử nhỏ nhất trong hai đầu còn lại, nên kết quả tăng dần và là hoán vị của hai nửa; phần dư được chép nguyên vẹn (đã sắp xếp và `>=` mọi phần tử đã chép).

**(c) Sắp xếp nhanh chia ba phần.** Sau vòng phân hoạch, `a[lo..lt-1] < p`, `a[lt..gt] == p`, `a[gt+1..hi] > p` (bất biến của cờ Hà Lan: mỗi bước chuyển một phần tử vào đúng vùng). Do mọi phần tử vùng `<` đều nhỏ hơn vùng `==` và vùng `>`, chỉ cần sắp xếp riêng hai vùng đó; các phần tử `== p` đã ở đúng vị trí cuối. Mỗi lần gọi loại ít nhất một phần tử (chốt) nên kết thúc.

**(d) Vun đống.** Sau khi dựng đống cực đại, `a[0]` là lớn nhất. Mỗi lần đổi `a[0]` với `a[end]` đặt phần tử lớn nhất còn lại vào đúng vị trí cuối của đoạn chưa sắp xếp; `sift_down(0, end)` khôi phục tính chất đống trên `a[0..end-1]` (chỉ nút gốc vi phạm, và `sift_down` sửa dọc theo một đường). Quy nạp theo `end`: sau khi xử lý `end`, đoạn `a[end..n-1]` là các phần tử lớn nhất theo thứ tự tăng dần.

**(e) Sắp xếp đếm.** `cnt[v]` bằng số lần giá trị `v - OFFSET` xuất hiện; duyệt `v` tăng dần và ghi lại đúng số lần cho từng giá trị tạo mảng không giảm là hoán vị của mảng gốc.

**Kiểm chứng bằng máy.** Mọi lời giải được so với oracle `sorted` (hàm sắp xếp có sẵn của Python) và với **bộ kiểm tra độc lập** `is_valid_sort` (không giảm **và** cùng đa tập phần tử), trên dữ liệu ngẫu nhiên và dữ liệu "khó" (đã sắp xếp, ngược, toàn bằng nhau, ít giá trị phân biệt, hình ống đàn organ, biên `+-5 * 10^4`). Lưu ý **chỉ phần kiểm thử** dùng `sorted`; các lời giải nộp không dùng.

In [20]:
def forbidden_calls_in(source: str):
    """Kiểm tra đơn giản: lời giải nộp không gọi hàm sắp xếp có sẵn."""
    banned = ("sorted(", ".sort(", "heapq", "bisect", "sort_values")
    return [b for b in banned if b in source]

import inspect
for name, cls in (("Cách 2", sol_merge), ("Cách 3", sol_quick), ("Cách 4", sol_heap), ("Cách 5", sol_count), ("nộp", Solution().sortArray)):
    src = inspect.getsource(cls.__func__ if hasattr(cls, "__func__") else cls)
    assert not forbidden_calls_in(src), (name, forbidden_calls_in(src))
print("mã nguồn các lời giải nộp (Cách 2-5) không dùng sorted / .sort / heapq / bisect")

mã nguồn các lời giải nộp (Cách 2-5) không dùng sorted / .sort / heapq / bisect


## 11. Time / Space Complexity

| Cách | Time (xấu nhất) | Space phụ | Ổn định |
|---|---|---|---|
| 1. Chèn | `O(n^2)` | `O(1)` | có |
| 2. Trộn | `O(n log n)` | `O(n)` | có |
| 3. Nhanh (3 phần, chốt ngẫu nhiên) | `O(n log n)` kỳ vọng | `O(log n)` | không |
| 4. Vun đống | `O(n log n)` | `O(1)` | không |
| 5. Đếm | `O(n + R)`, `R = 100001` | `O(R)` | có (nếu viết kèm dữ liệu) |

Đo thử với `n = 5 * 10^4` (cỡ tối đa) trên nhiều dạng dữ liệu (chỉ chạy `O(n^2)` ở cỡ nhỏ):

In [21]:
rng = random.Random(11)
n = 5 * 10 ** 4
datasets = {
    "ngẫu nhiên": [rng.randint(-50000, 50000) for _ in range(n)],
    "đã sắp xếp": list(range(-25000, 25000)),
    "ngược": list(range(25000, -25000, -1)),
    "toàn bằng nhau": [7] * n,
    "ít giá trị (5)": [rng.randint(0, 4) for _ in range(n)],
}
for dname, data in datasets.items():
    ref = sorted(data)
    row = []
    for name, fn in (("trộn", sol_merge), ("nhanh", sol_quick), ("vun đống", sol_heap), ("đếm", sol_count)):
        arr = list(data)
        t0 = time.perf_counter()
        out = fn(arr)
        row.append(f"{name}: {1000 * (time.perf_counter() - t0):7.1f} ms")
        assert out == ref, (dname, name)
    print(f"{dname:16} " + " | ".join(row))

ngẫu nhiên       trộn:   101.7 ms | nhanh:   130.4 ms | vun đống:   206.8 ms | đếm:    14.7 ms


đã sắp xếp       trộn:   123.7 ms | nhanh:   113.6 ms | vun đống:   132.8 ms | đếm:     8.1 ms


ngược            trộn:    78.7 ms | nhanh:   122.7 ms | vun đống:   125.7 ms | đếm:     8.2 ms
toàn bằng nhau   trộn:    79.0 ms | nhanh:     3.1 ms | vun đống:    12.0 ms | đếm:     8.3 ms


ít giá trị (5)   trộn:    90.4 ms | nhanh:    11.5 ms | vun đống:   101.9 ms | đếm:     8.2 ms


## 12. Bộ test

Gồm: ví dụ của đề, edge case, phản ví dụ cho các cách sai (tìm bằng tìm kiếm có seed hoặc đo số phép so sánh), đối chiếu mọi cách với oracle `sorted` **và** bộ kiểm tra độc lập `is_valid_sort`, test ngẫu nhiên có seed (đúng ràng buộc) và các dữ liệu khó.

In [22]:
solutions = {
    "insertion": insertion_sort,
    "merge": merge_sort,
    "quick": quick_sort,
    "heap": heap_sort,
    "counting": counting_sort,
    "sol_merge": sol_merge,
    "sol_quick": sol_quick,
    "sol_heap": sol_heap,
    "sol_count": sol_count,
    "final": Solution().sortArray,
}

def run_all(nums, skip=()):
    ref = sorted(nums)                                   # oracle (chỉ trong kiểm thử)
    for name, fn in solutions.items():
        if name in skip:
            continue
        arr = list(nums)
        out = fn(arr)
        assert out == ref, (name, nums[:10], len(nums))
        assert is_valid_sort(nums, out), (name, "bộ kiểm tra độc lập")
    return ref

# 12.1 ví dụ đề
for nums, expected in examples:
    assert run_all(nums) == expected
print("12.1 ví dụ đề: OK")

# 12.2 edge case (kỳ vọng viết tay, kèm lý do)
edge = [
    ([1], [1]),                                          # một phần tử
    ([2, 1], [1, 2]),                                    # hai phần tử đảo
    ([1, 1], [1, 1]),                                    # trùng
    ([3, 3, 3, 3], [3, 3, 3, 3]),                        # toàn bằng nhau
    ([-1, -2, -3], [-3, -2, -1]),                        # toàn âm
    ([0, -50000, 50000], [-50000, 0, 50000]),            # biên của đề
    ([50000, -50000, 50000, -50000], [-50000, -50000, 50000, 50000]),
    ([1, 2, 3, 4, 5], [1, 2, 3, 4, 5]),                  # đã sắp xếp
    ([5, 4, 3, 2, 1], [1, 2, 3, 4, 5]),                  # ngược
    ([2, 1, 2, 1, 2, 1], [1, 1, 1, 2, 2, 2]),            # xen kẽ ít giá trị
    ([0, 0, 0, -1, -1, 1, 1], [-1, -1, 0, 0, 0, 1, 1]),
]
for nums, expected in edge:
    assert run_all(nums) == expected, nums
print("12.2 edge case: OK", len(edge), "ca")

12.1 ví dụ đề: OK
12.2 edge case: OK 11 ca


In [23]:
# 12.3 phản ví dụ: các cách sai
comparisons = 0

def quick_sort_lomuto_last_pivot(nums: List[int]) -> List[int]:
    """Quick sort Lomuto, chốt luôn là phần tử CUỐI, đếm phép so sánh (dạng bài hay gặp)."""
    global comparisons
    a = nums
    def qs(lo, hi):
        global comparisons
        if lo >= hi:
            return
        p = a[hi]
        i = lo
        for j in range(lo, hi):
            comparisons += 1
            if a[j] < p:
                a[i], a[j] = a[j], a[i]
                i += 1
        a[i], a[hi] = a[hi], a[i]
        qs(lo, i - 1)
        qs(i + 1, hi)
    qs(0, len(a) - 1)
    return a

def merge_sort_no_leftover(nums):                        # quên chép phần dư khi một nửa hết
    n = len(nums)
    tmp = [0] * n
    def sort(lo, hi):
        if hi - lo <= 1:
            return
        mid = (lo + hi) // 2
        sort(lo, mid)
        sort(mid, hi)
        i, j, k = lo, mid, lo
        while i < mid and j < hi:
            if nums[i] <= nums[j]:
                tmp[k] = nums[i]; i += 1
            else:
                tmp[k] = nums[j]; j += 1
            k += 1
        nums[lo:k] = tmp[lo:k]                           # thiếu hai vòng chép phần dư
    sort(0, n)
    return nums

def heap_sort_wrong_bound(nums):                         # sift_down dùng len(a) thay cho size
    a = nums
    n = len(a)
    def sift_down(i, size):
        size = len(a)                                    # SAI: bỏ qua biên đống hiện tại
        while True:
            left = 2 * i + 1
            if left >= size:
                return
            c = left
            if left + 1 < size and a[left + 1] > a[left]:
                c = left + 1
            if a[c] > a[i]:
                a[c], a[i] = a[i], a[c]
                i = c
            else:
                return
    for i in range(n // 2 - 1, -1, -1):
        sift_down(i, n)
    for end in range(n - 1, 0, -1):
        a[0], a[end] = a[end], a[0]
        sift_down(0, end)
    return a

def counting_sort_no_offset(nums):                       # quên dịch chỉ số âm: cnt[x] với x < 0 ghi vào cuối mảng
    cnt = [0] * 100001
    for x in nums:
        cnt[x] += 1
    out = []
    for v in range(100001):
        out.extend([v] * cnt[v])
    return out

def find_counter(fn, tries=20000, seed=5):
    rng = random.Random(seed)
    for _ in range(tries):
        n = rng.randint(1, 8)
        nums = [rng.randint(-5, 5) for _ in range(n)]
        try:
            out = fn(list(nums))
        except Exception:
            return nums
        if out != sorted(nums):
            return nums
    return None

counters = {}
for name, fn in (("merge thiếu phần dư", merge_sort_no_leftover), ("heap sai biên đống", heap_sort_wrong_bound),
                 ("counting không dịch chỉ số âm", counting_sort_no_offset)):
    nums = find_counter(fn)
    assert nums is not None, name
    counters[name] = nums
    try:
        got = fn(list(nums))
    except Exception as e:
        got = f"lỗi {type(e).__name__}"
    print(f"{name:30} nums={nums}: sai -> {got}, đúng -> {sorted(nums)}")
    run_all(nums)
print("12.3 phản ví dụ (sai kết quả): OK")

merge thiếu phần dư            nums=[0, 5, 3, -5, 2]: sai -> [-5, 0, 2, 2, 2], đúng -> [-5, 0, 2, 3, 5]
heap sai biên đống             nums=[0, 5, 3, -5, 2]: sai -> [5, 3, 2, 0, -5], đúng -> [-5, 0, 2, 3, 5]
counting không dịch chỉ số âm  nums=[0, 5, 3, -5, 2]: sai -> [0, 2, 3, 5, 99996], đúng -> [-5, 0, 2, 3, 5]
12.3 phản ví dụ (sai kết quả): OK


In [24]:
# 12.3b quick sort chốt cố định trên dữ liệu đã sắp xếp: đúng kết quả nhưng O(n^2) và tràn đệ quy
for n in (100, 200, 400, 800):
    comparisons = 0
    quick_sort_lomuto_last_pivot(list(range(n)))
    print(f"n={n:4}: {comparisons:7} phép so sánh (n^2/2 = {n * n // 2}, n log2 n ~ {int(n * n.bit_length())})")

old = sys.getrecursionlimit()
sys.setrecursionlimit(1000)
try:
    quick_sort_lomuto_last_pivot(list(range(3000)))
    print("không lỗi?")
except RecursionError as e:
    print("RecursionError trên dữ liệu đã sắp xếp n = 3000:", e)
finally:
    sys.setrecursionlimit(old)
print("bản nộp (chốt ngẫu nhiên, 3 phần) trên cùng dữ liệu:", Solution().sortArray(list(range(3000)))[:5], "...")
assert Solution().sortArray(list(range(3000))) == list(range(3000))
print("12.3b: OK")

n= 100:    4950 phép so sánh (n^2/2 = 5000, n log2 n ~ 700)
n= 200:   19900 phép so sánh (n^2/2 = 20000, n log2 n ~ 1600)
n= 400:   79800 phép so sánh (n^2/2 = 80000, n log2 n ~ 3600)


n= 800:  319600 phép so sánh (n^2/2 = 320000, n log2 n ~ 8000)


RecursionError trên dữ liệu đã sắp xếp n = 3000: maximum recursion depth exceeded
bản nộp (chốt ngẫu nhiên, 3 phần) trên cùng dữ liệu: [0, 1, 2, 3, 4] ...
12.3b: OK


In [25]:
# 12.4 ngẫu nhiên nhỏ: mọi cách (gồm sắp xếp chèn) so với oracle và bộ kiểm tra độc lập
rng = random.Random(4912)
for _ in range(2000):
    n = rng.randint(1, 30)
    lo, hi = rng.choice([(0, 3), (-5, 5), (-50000, 50000)])
    nums = [rng.randint(lo, hi) for _ in range(n)]
    run_all(nums)
print("12.4 ngẫu nhiên n <= 30 (2000 ca): OK")

# 12.5 trung bình: n tới 3000 (kể cả sắp xếp chèn)
rng = random.Random(5912)
for _ in range(60):
    n = rng.randint(31, 3000)
    lo, hi = rng.choice([(0, 5), (-100, 100), (-50000, 50000)])
    nums = [rng.randint(lo, hi) for _ in range(n)]
    run_all(nums)
print("12.5 ngẫu nhiên n tới 3000 (60 ca): OK")

# 12.6 lớn n = 5 * 10^4 (đúng ràng buộc) và dữ liệu khó; bỏ sắp xếp chèn
rng = random.Random(6912)
n = 5 * 10 ** 4
hard = {
    "ngẫu nhiên": [rng.randint(-50000, 50000) for _ in range(n)],
    "đã sắp xếp": list(range(-25000, 25000)),
    "ngược": list(range(25000, -25000, -1)),
    "toàn bằng nhau": [-50000] * n,
    "hai giá trị": [rng.choice((-50000, 50000)) for _ in range(n)],
    "hình ống đàn organ": list(range(0, 25000)) + list(range(25000, 0, -1)),
    "xen kẽ lớn nhỏ": [(-1) ** i * (i // 2) for i in range(n)],
    "toàn giá trị khác nhau xáo trộn": rng.sample(range(-50000, 50001), n),
}
for name, data in hard.items():
    assert len(data) <= n and all(-50000 <= x <= 50000 for x in data), name
    run_all(data, skip=("insertion",))
    print(f"  {name:34} OK")
print("12.6 n = 5 * 10^4 và dữ liệu khó: OK")

# 12.7 tính chất: bất biến theo hoán vị đầu vào; sắp xếp hai lần cho cùng kết quả; sắp xếp trộn ổn định
rng = random.Random(7912)
for _ in range(200):
    nums = [rng.randint(-20, 20) for _ in range(rng.randint(1, 40))]
    shuffled = nums[:]
    rng.shuffle(shuffled)
    a = Solution().sortArray(list(nums))
    assert Solution().sortArray(shuffled) == a and Solution().sortArray(list(a)) == a
# tính ổn định của sắp xếp trộn (kiểm tra bằng cặp (giá trị, thứ tự ban đầu))
def stable_merge_pairs(pairs):
    tmp = [None] * len(pairs)
    def sort(lo, hi):
        if hi - lo <= 1:
            return
        mid = (lo + hi) // 2
        sort(lo, mid); sort(mid, hi)
        i, j, k = lo, mid, lo
        while i < mid and j < hi:
            if pairs[i][0] <= pairs[j][0]:
                tmp[k] = pairs[i]; i += 1
            else:
                tmp[k] = pairs[j]; j += 1
            k += 1
        while i < mid:
            tmp[k] = pairs[i]; i += 1; k += 1
        while j < hi:
            tmp[k] = pairs[j]; j += 1; k += 1
        pairs[lo:hi] = tmp[lo:hi]
    sort(0, len(pairs))
    return pairs

for _ in range(200):
    vals = [rng.randint(0, 4) for _ in range(rng.randint(1, 40))]
    pairs = stable_merge_pairs([(v, idx) for idx, v in enumerate(vals)])
    assert all(pairs[i][0] < pairs[i + 1][0] or (pairs[i][0] == pairs[i + 1][0] and pairs[i][1] < pairs[i + 1][1]) for i in range(len(pairs) - 1))
print("12.7 tính chất (bất biến, lũy đẳng, ổn định của trộn): OK")

12.4 ngẫu nhiên n <= 30 (2000 ca): OK


12.5 ngẫu nhiên n tới 3000 (60 ca): OK


  ngẫu nhiên                         OK


  đã sắp xếp                         OK


  ngược                              OK


  toàn bằng nhau                     OK


  hai giá trị                        OK


  hình ống đàn organ                 OK


  xen kẽ lớn nhỏ                     OK


  toàn giá trị khác nhau xáo trộn    OK
12.6 n = 5 * 10^4 và dữ liệu khó: OK
12.7 tính chất (bất biến, lũy đẳng, ổn định của trộn): OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Quick sort với chốt cố định trên dữ liệu đã sắp xếp

Chọn chốt luôn là phần tử cuối (hoặc đầu) trên mảng đã sắp xếp chia rất lệch: một phần rỗng, một phần `n - 1`: `O(n^2)` thời gian và đệ quy sâu `n`.

In [26]:
comparisons = 0
quick_sort_lomuto_last_pivot(list(range(800)))
c_sorted = comparisons
comparisons = 0
rng = random.Random(1)
arr = list(range(800))
rng.shuffle(arr)
quick_sort_lomuto_last_pivot(arr)
c_random = comparisons
print(f"n=800: dữ liệu đã sắp xếp {c_sorted} phép so sánh, dữ liệu xáo trộn {c_random} phép so sánh")
assert c_sorted > 10 * c_random

n=800: dữ liệu đã sắp xếp 319600 phép so sánh, dữ liệu xáo trộn 8201 phép so sánh


Vì sao sai: chốt không chia đôi dữ liệu. **Đúng:** chốt **ngẫu nhiên** (hoặc trung vị của ba), chia **ba phần** để xử lý số trùng, đệ quy vào phần nhỏ hơn; hoặc dùng thuật toán `O(n log n)` đảm bảo (heap sort, merge sort).

### Lỗi 2 — Trộn mà quên chép phần dư

Sau vòng `while i < mid and j < hi`, một trong hai nửa còn phần tử chưa chép. Bỏ qua chúng làm mất phần tử.

In [27]:
nums = counters["merge thiếu phần dư"]
print("nums =", nums, "| sai ->", merge_sort_no_leftover(list(nums)), "| đúng ->", Solution().sortArray(list(nums)))
assert merge_sort_no_leftover(list(nums)) != sorted(nums)

nums = [0, 5, 3, -5, 2] | sai -> [-5, 0, 2, 2, 2] | đúng -> [-5, 0, 2, 3, 5]


Vì sao sai: nửa còn lại chứa các phần tử lớn nhất chưa được đặt vào `tmp`. **Đúng:** thêm `while i < mid` và `while j < hi` để chép phần dư.

### Lỗi 3 — Sai biên đống khi vun đống

`sift_down(0, end)` phải coi đống chỉ gồm `end` phần tử đầu; nếu dùng `len(a)` thì phần đuôi đã sắp xếp bị kéo lại vào đống và bị phá.

In [28]:
nums = counters["heap sai biên đống"]
print("nums =", nums, "| sai ->", heap_sort_wrong_bound(list(nums)), "| đúng ->", Solution().sortArray(list(nums)))
assert heap_sort_wrong_bound(list(nums)) != sorted(nums)

nums = [0, 5, 3, -5, 2] | sai -> [5, 3, 2, 0, -5] | đúng -> [-5, 0, 2, 3, 5]


Vì sao sai: các phần tử lớn đã đặt đúng chỗ ở cuối bị `sift_down` đổi lại. **Đúng:** truyền kích thước đống hiện tại `size = end`.

### Lỗi 4 — Counting sort quên dịch chỉ số âm

Python cho phép chỉ số **âm** (đếm từ cuối), nên `cnt[x]` với `x < 0` **không lỗi** mà ghi vào ô sai, cho kết quả sai âm thầm.

In [29]:
nums = counters["counting không dịch chỉ số âm"]
print("nums =", nums, "| sai ->", counting_sort_no_offset(list(nums)), "| đúng ->", Solution().sortArray(list(nums)))
assert counting_sort_no_offset(list(nums)) != sorted(nums)

nums = [0, 5, 3, -5, 2] | sai -> [0, 2, 3, 5, 99996] | đúng -> [-5, 0, 2, 3, 5]


Vì sao sai: `cnt[-3]` là phần tử thứ ba từ cuối chứ không phải "giá trị `-3`". **Đúng:** `cnt[x + 50000]`.

### Lỗi 5 — Dùng thuật toán `O(n^2)` với `n = 5 * 10^4`

Sắp xếp chèn/chọn/nổi bọt cần `~ n^2 / 4 = 6 * 10^8` thao tác: quá chậm (xem mục 5).

In [30]:
rng = random.Random(5)
for n in (1000, 2000, 4000):
    nums = [rng.randint(-50000, 50000) for _ in range(n)]
    t0 = time.perf_counter(); insertion_sort(list(nums)); t_ins = time.perf_counter() - t0
    t0 = time.perf_counter(); Solution().sortArray(list(nums)); t_heap = time.perf_counter() - t0
    print(f"n={n:5}: chèn {1000 * t_ins:8.1f} ms | vun đống {1000 * t_heap:6.1f} ms | tỷ lệ {t_ins / t_heap:5.1f}x")

n= 1000: chèn     24.3 ms | vun đống    2.2 ms | tỷ lệ  11.2x


n= 2000: chèn     85.2 ms | vun đống    3.8 ms | tỷ lệ  22.6x


n= 4000: chèn    331.0 ms | vun đống   10.6 ms | tỷ lệ  31.2x


Vì sao chậm: thời gian chèn tăng bình phương theo `n`. **Đúng:** thuật toán `O(n log n)`.

## 14. Mẹo & nhận dạng pattern

- **Dấu hiệu:** "tự cài đặt sắp xếp" ⇒ biết bốn thuật toán `O(n log n)`: **trộn**, **nhanh**, **vun đống** và (khi miền nhỏ) **đếm**.
- **Bộ nhớ phụ nhỏ nhất:** vun đống `O(1)` > nhanh `O(log n)` > trộn `O(n)`.
- **Quick sort an toàn:** chốt ngẫu nhiên + chia ba phần + đệ quy vào phần nhỏ hơn.
- **Tính ổn định:** trộn và đếm (khi viết kèm dữ liệu) ổn định; nhanh và vun đống thì không.
- **Miền giá trị nhỏ:** đếm / bucket / radix sort `O(n)`.
- Cùng họ: Sort Colors (75), Kth Largest Element in an Array (215), Merge Sorted Array (88), Count of Smaller Numbers After Self (315, dùng trộn), Reverse Pairs (493).
- Test nên có: một phần tử, toàn bằng nhau, đã sắp xếp, ngược, ít giá trị phân biệt, hình ống đàn organ, biên `+-5 * 10^4`.

## 15. Biến thể

1. **Đếm nghịch thế (inversions)** bằng sắp xếp trộn: số cặp `i < j` có `a[i] > a[j]` (bài tập 5).
2. **Phần tử lớn thứ `k`** bằng chọn nhanh (quickselect) `O(n)` kỳ vọng.
3. **Sắp xếp ổn định theo khóa** (sắp xếp cặp theo giá trị): dùng trộn.
4. Bài liên quan: Kth Largest Element in an Array (215), Count of Smaller Numbers After Self (315).

Dưới đây là biến thể 2 (quickselect) kiểm bằng `sorted`.

In [31]:
def quickselect_kth_largest(nums: List[int], k: int) -> int:
    a = list(nums)
    target = len(a) - k                                   # chỉ số trong mảng tăng dần
    lo, hi = 0, len(a) - 1
    while True:
        p = a[random.randint(lo, hi)]
        lt, i, gt = lo, lo, hi
        while i <= gt:
            if a[i] < p:
                a[lt], a[i] = a[i], a[lt]; lt += 1; i += 1
            elif a[i] > p:
                a[i], a[gt] = a[gt], a[i]; gt -= 1
            else:
                i += 1
        if target < lt:
            hi = lt - 1
        elif target > gt:
            lo = gt + 1
        else:
            return p

rng = random.Random(15)
for _ in range(500):
    n = rng.randint(1, 30)
    nums = [rng.randint(-10, 10) for _ in range(n)]
    k = rng.randint(1, n)
    assert quickselect_kth_largest(nums, k) == sorted(nums, reverse=True)[k - 1], (nums, k)
print("quickselect khớp sorted; ví dụ:", quickselect_kth_largest([3, 2, 1, 5, 6, 4], 2))

quickselect khớp sorted; ví dụ: 5


## 16. So sánh các cách

| Cách | Ý tưởng | Time xấu nhất | Space phụ | Ổn định | Khi nào dùng |
|---|---|---|---|---|---|
| 1. Chèn | Dịch từng bước | `O(n^2)` | `O(1)` | có | Mảng nhỏ / gần sắp xếp |
| 2. Trộn | Chia đôi và trộn | `O(n log n)` | `O(n)` | có | Cần ổn định, dữ liệu liên kết |
| 3. Nhanh | Chia ba theo chốt ngẫu nhiên | `O(n log n)` kỳ vọng | `O(log n)` | không | Nhanh thực tế |
| 4. Vun đống | Đống cực đại | `O(n log n)` | `O(1)` | không | **Cách nộp chính** (đề yêu cầu bộ nhớ nhỏ nhất) |
| 5. Đếm | Đếm theo giá trị | `O(n + R)` | `O(R)` | có | Miền giá trị nhỏ |

## 17. Bài tập vận dụng

Làm tuần tự từ dễ đến khó. Các cell có chữ `TODO` **không được gọi** khi chạy notebook; bạn tự điền rồi bỏ dấu `#` ở các dòng `assert` để kiểm tra. Đáp án nằm ở mục 19.

### Bài 1 — Đoán output (dễ)

Không chạy code, hãy đoán output ba dòng, rồi chạy để kiểm tra:

```python
print(merge([1, 4], [2, 3]))
print((3 - 1) // 2, 2 * 3 + 1, 2 * 3 + 2)
print(is_valid_sort([2, 1, 2], [1, 2, 2]), is_valid_sort([2, 1, 2], [1, 1, 2]))
```

In [32]:
# Bài 1: ghi dự đoán của bạn
my_guess_line1 = None
my_guess_line2 = None
my_guess_line3 = None

### Bài 2 — Tính tay (dễ-vừa)

Chạy tay **sắp xếp chèn** cho `[5, 2, 4, 1]`: ghi mảng sau mỗi lần chèn `i = 1, 2, 3`.

In [33]:
# Bài 2: điền kết quả tính tay
hand_passes = None            # danh sách 3 mảng

### Bài 3 — Điền chỗ trống (vừa)

Hoàn thành hàm trộn hai mảng đã sắp xếp.

In [34]:
def exercise_3_merge(L: List[int], R: List[int]) -> List[int]:
    out, i, j = [], 0, 0
    while i < len(L) and j < len(R):
        # TODO 1: lấy phần tử nhỏ hơn (khi bằng nhau lấy bên trái), tăng con trỏ tương ứng
        pass
    # TODO 2: chép phần dư của L và của R
    return out


# assert exercise_3_merge([1, 4], [2, 3]) == [1, 2, 3, 4]
# assert exercise_3_merge([2, 2], [1, 2, 3]) == [1, 2, 2, 2, 3]
# assert exercise_3_merge([], [1]) == [1]

### Bài 4 — Tự viết lại (vừa-khó)

Không nhìn lại Cách 5, hãy tự viết `exercise_4_counting(nums)` (sắp xếp đếm có dịch chỉ số, miền `[-5 * 10^4, 5 * 10^4]`), không dùng `sorted`.

In [35]:
def exercise_4_counting(nums: List[int]) -> List[int]:
    # TODO
    pass


# assert exercise_4_counting([5, 1, 1, 2, 0, 0, -3]) == [-3, 0, 0, 1, 1, 2, 5]
# assert exercise_4_counting([3]) == [3]

### Bài 5 — Mini challenge: đếm nghịch thế (khó)

Viết `exercise_5_inversions(nums)` trả về **số cặp** `(i, j)` với `i < j` và `nums[i] > nums[j]` trong `O(n log n)` bằng sắp xếp trộn (không dùng `sorted`).

```python
exercise_5_inversions([2, 4, 1, 3, 5])  -> 3     # (2,1), (4,1), (4,3)
exercise_5_inversions([1, 2, 3])        -> 0
exercise_5_inversions([3, 2, 1])        -> 3
```
Gợi ý: khi trộn, mỗi lần lấy phần tử bên **phải** trước phần tử bên trái còn lại thì có `mid - i` nghịch thế mới.

In [36]:
def exercise_5_inversions(nums: List[int]) -> int:
    # TODO
    pass


# assert exercise_5_inversions([2, 4, 1, 3, 5]) == 3
# assert exercise_5_inversions([1, 2, 3]) == 0
# assert exercise_5_inversions([3, 2, 1]) == 3

## 18. Gợi ý 3 mức

**Bài 1**
- Mức 1: `merge` trộn hai danh sách đã sắp xếp; chỉ số đống: cha `(i - 1) // 2`, con `2i + 1`, `2i + 2`.
- Mức 2: `merge([1,4],[2,3])` lấy `1`, rồi `2`, rồi `3`, rồi phần dư `4`.
- Mức 3: `[1, 2, 3, 4]`, `1 7 8`, `True False`.

**Bài 2**
- Mức 1: `[5, 2, 4, 1]`: chèn `2` vào `[5]`.
- Mức 2: `i = 1` cho `[2, 5, 4, 1]`; `i = 2` chèn `4` giữa `2` và `5`.
- Mức 3: `[2, 5, 4, 1]`, `[2, 4, 5, 1]`, `[1, 2, 4, 5]`.

**Bài 3**
- Mức 1: nếu `L[i] <= R[j]` thì thêm `L[i]`, `i += 1`; ngược lại thêm `R[j]`, `j += 1`.
- Mức 2: sau vòng lặp, một trong hai nửa còn phần tử.
- Mức 3: `out.extend(L[i:])`; `out.extend(R[j:])`.

**Bài 4**
- Mức 1: `cnt = [0] * 100001`; `cnt[x + 50000] += 1`.
- Mức 2: duyệt `v` từ `0` tới `100000`.
- Mức 3: ghi `v - 50000` đúng `cnt[v]` lần vào kết quả.

**Bài 5**
- Mức 1: dùng khung merge sort; đếm trong bước trộn.
- Mức 2: khi `L[i] > R[j]` (lấy bên phải) thì mọi phần tử còn lại `L[i..]` đều lớn hơn `R[j]`: cộng `len(L) - i`.
- Mức 3: tổng = nghịch thế của hai nửa + nghịch thế khi trộn; trả về tổng.

## 19. Đáp án tham khảo

Bạn nên tự làm trước khi xem phần này. Mỗi cell dưới đây đều tự kiểm tra bằng `assert`.

### Đáp án Bài 1

In [37]:
print(merge([1, 4], [2, 3]))
print((3 - 1) // 2, 2 * 3 + 1, 2 * 3 + 2)
print(is_valid_sort([2, 1, 2], [1, 2, 2]), is_valid_sort([2, 1, 2], [1, 1, 2]))

assert merge([1, 4], [2, 3]) == [1, 2, 3, 4]
assert ((3 - 1) // 2, 2 * 3 + 1, 2 * 3 + 2) == (1, 7, 8)
assert (is_valid_sort([2, 1, 2], [1, 2, 2]), is_valid_sort([2, 1, 2], [1, 1, 2])) == (True, False)

[1, 2, 3, 4]
1 7 8
True False


### Đáp án Bài 2

`i = 1`: chèn `2` -> `[2, 5, 4, 1]`; `i = 2`: chèn `4` -> `[2, 4, 5, 1]`; `i = 3`: chèn `1` -> `[1, 2, 4, 5]`.

In [38]:
def insertion_passes(nums):
    a = list(nums)
    passes = []
    for i in range(1, len(a)):
        key, j = a[i], i - 1
        while j >= 0 and a[j] > key:
            a[j + 1] = a[j]
            j -= 1
        a[j + 1] = key
        passes.append(list(a))
    return passes

hand_passes_solution = [[2, 5, 4, 1], [2, 4, 5, 1], [1, 2, 4, 5]]
assert insertion_passes([5, 2, 4, 1]) == hand_passes_solution
print("Bài 2 OK")

Bài 2 OK


### Đáp án Bài 3

In [39]:
def exercise_3_merge_solution(L: List[int], R: List[int]) -> List[int]:
    out, i, j = [], 0, 0
    while i < len(L) and j < len(R):
        if L[i] <= R[j]:
            out.append(L[i])
            i += 1
        else:
            out.append(R[j])
            j += 1
    out.extend(L[i:])
    out.extend(R[j:])
    return out

assert exercise_3_merge_solution([1, 4], [2, 3]) == [1, 2, 3, 4]
assert exercise_3_merge_solution([2, 2], [1, 2, 3]) == [1, 2, 2, 2, 3]
assert exercise_3_merge_solution([], [1]) == [1]
rng = random.Random(3)
for _ in range(300):
    A = sorted(rng.randint(-5, 5) for _ in range(rng.randint(0, 8)))
    B = sorted(rng.randint(-5, 5) for _ in range(rng.randint(0, 8)))
    assert exercise_3_merge_solution(A, B) == sorted(A + B)
print("Bài 3 OK")

Bài 3 OK


### Đáp án Bài 4

In [40]:
def exercise_4_counting_solution(nums: List[int]) -> List[int]:
    cnt = [0] * 100001
    for x in nums:
        cnt[x + 50000] += 1
    out = []
    for v in range(100001):
        out.extend([v - 50000] * cnt[v])
    return out

assert exercise_4_counting_solution([5, 1, 1, 2, 0, 0, -3]) == [-3, 0, 0, 1, 1, 2, 5]
assert exercise_4_counting_solution([3]) == [3]
rng = random.Random(4)
for _ in range(200):
    nums = [rng.randint(-50000, 50000) for _ in range(rng.randint(1, 50))]
    assert exercise_4_counting_solution(nums) == sorted(nums)
print("Bài 4 OK")

Bài 4 OK


### Đáp án Bài 5

In [41]:
def exercise_5_inversions_solution(nums: List[int]) -> int:
    a = list(nums)
    tmp = [0] * len(a)

    def sort(lo, hi):
        if hi - lo <= 1:
            return 0
        mid = (lo + hi) // 2
        inv = sort(lo, mid) + sort(mid, hi)
        i, j, k = lo, mid, lo
        while i < mid and j < hi:
            if a[i] <= a[j]:
                tmp[k] = a[i]
                i += 1
            else:
                tmp[k] = a[j]
                j += 1
                inv += mid - i                        # mọi a[i..mid-1] đều lớn hơn a[j]
            k += 1
        while i < mid:
            tmp[k] = a[i]; i += 1; k += 1
        while j < hi:
            tmp[k] = a[j]; j += 1; k += 1
        a[lo:hi] = tmp[lo:hi]
        return inv

    return sort(0, len(a))

def inversions_brute(nums):
    return sum(1 for i in range(len(nums)) for j in range(i + 1, len(nums)) if nums[i] > nums[j])

assert exercise_5_inversions_solution([2, 4, 1, 3, 5]) == 3
assert exercise_5_inversions_solution([1, 2, 3]) == 0
assert exercise_5_inversions_solution([3, 2, 1]) == 3
rng = random.Random(5)
for _ in range(300):
    nums = [rng.randint(-5, 5) for _ in range(rng.randint(0, 30))]
    assert exercise_5_inversions_solution(nums) == inversions_brute(nums), nums
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

### Template (vun đống: `O(n log n)` xấu nhất, `O(1)` bộ nhớ phụ)

```python
class Solution:
    def sortArray(self, nums: List[int]) -> List[int]:
        n = len(nums)

        def sift_down(i: int, size: int) -> None:          # đưa nums[i] xuống đúng chỗ trong đống nums[:size]
            while True:
                left = 2 * i + 1
                if left >= size:
                    return
                c = left
                if left + 1 < size and nums[left + 1] > nums[left]:
                    c = left + 1                            # chọn con lớn hơn
                if nums[c] > nums[i]:
                    nums[c], nums[i] = nums[i], nums[c]
                    i = c
                else:
                    return

        for i in range(n // 2 - 1, -1, -1):                 # dựng đống cực đại
            sift_down(i, n)
        for end in range(n - 1, 0, -1):                     # đưa lớn nhất ra cuối
            nums[0], nums[end] = nums[end], nums[0]
            sift_down(0, end)                                # biên đống = end
        return nums
```

Công thức nhớ nhanh: con trái `2i + 1`, con phải `2i + 2`; dựng đống từ `n // 2 - 1` về `0`; sau mỗi lần đổi gốc với `end`, vun lại trong `[0, end)`.

### Checklist trước khi submit

- [ ] Không gọi `sorted`, `list.sort`, `heapq`... trong lời giải nộp?
- [ ] Thuật toán `O(n log n)` trong **mọi** trường hợp (hoặc chốt ngẫu nhiên + chia ba phần)?
- [ ] Trộn: chép **phần dư**; vun đống: biên đống `size = end` (không dùng `len`)?
- [ ] Nếu dùng đếm: dịch chỉ số `x + 50000` cho số âm?
- [ ] Đã thử: một phần tử, toàn bằng nhau, đã sắp xếp, ngược, ít giá trị phân biệt, biên `+-5 * 10^4`, `n = 5 * 10^4`?
- [ ] Tên method đúng `sortArray(self, nums)` và trả `List[int]`?